In [12]:
# ──── Imports ────
import os
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModel
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from torch.amp.autocast_mode import autocast 
from torch.amp.grad_scaler import GradScaler
from torch.utils.data import Subset

In [13]:
# ──── Configs ────
DATASET_DIR = "D:/Projects/Masters/Data/AR_merged_dataset"

# ──── Hyperparameters ────
BATCH_SIZE   = 32   # bumped from 16 — more stable gradient estimates
NUM_WORKERS  = 2
IMAGE_SIZE   = 224
MAX_EPOCHS = 10
# Unfreezing schedule (epoch numbers, 1-indexed)
UNFREEZE_LAST_N_EPOCH = 4   # unfreeze last 2 transformer blocks at this epoch
UNFREEZE_FULL_EPOCH   = 8  # unfreeze full backbone at this epoch

# Early stopping
PATIENCE = 8  # stop if val F1 doesn't improve for this many epochs

SAVE_PATH = "best_dinov2_ar.pt"

In [14]:
# ──── Class definitions & helper functions ────
class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, num_classes, hidden_size=384):
        super().__init__()
        self.backbone   = backbone
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),           # GELU fits better with transformer backbones than ReLU
            nn.Dropout(0.4),     
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.3),     
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        outputs   = self.backbone(x)
        cls_token = outputs.last_hidden_state[:, 0]
        return self.classifier(cls_token)


def freeze_backbone(model):
    """Freeze entire backbone."""
    for param in model.backbone.parameters():
        param.requires_grad = False


def unfreeze_last_n_blocks(model, n=2):
    """Unfreeze only the last n transformer blocks + layernorm."""
    # First make sure everything is frozen
    freeze_backbone(model)
    # DINOv2 encoder blocks live at model.backbone.encoder.layer
    blocks = model.backbone.encoder.layer
    for block in blocks[-n:]:
        for param in block.parameters():
            param.requires_grad = True
    # Also unfreeze the final layernorm
    for param in model.backbone.layernorm.parameters():
        param.requires_grad = True


def unfreeze_full_backbone(model):
    """Unfreeze everything."""
    for param in model.backbone.parameters():
        param.requires_grad = True

In [15]:
# ──── Transformers & Augmentations ────
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [16]:
train_dataset = datasets.ImageFolder(
    root=os.path.join(DATASET_DIR, "train"),
    transform=train_transform
)
val_dataset = datasets.ImageFolder(
    root=os.path.join(DATASET_DIR, "val"),
    transform=val_transform
)


print(f"Classes       : {train_dataset.classes}")
print(f"Class mapping : {train_dataset.class_to_idx}")
print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")

# ──── Class weights ────
train_labels  = [label for _, label in train_dataset.samples]
label_counts  = np.bincount(train_labels)
class_weights = label_counts.sum() / (len(label_counts) * label_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

print("\nClass weights (to balance loss):")
for cls, w in zip(train_dataset.classes, class_weights):
    print(f"  {cls:15s}: {w:.4f}")

# ──── Distribution check ────
print("\nTrain distribution:")
for idx, cls in enumerate(train_dataset.classes):
    n   = label_counts[idx]
    pct = 100 * n / len(train_labels)
    print(f"  {cls:15s}: {n:>6}  ({pct:.1f}%)")

Classes       : ['fanning', 'neutral', 'trophallaxis']
Class mapping : {'fanning': 0, 'neutral': 1, 'trophallaxis': 2}
Train samples : 88260
Val samples   : 17634

Class weights (to balance loss):
  fanning        : 0.6158
  neutral        : 1.3333
  trophallaxis   : 1.5970

Train distribution:
  fanning        :  47773  (54.1%)
  neutral        :  22065  (25.0%)
  trophallaxis   :  18422  (20.9%)


In [17]:
# ──── Dataloaders ────
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False
)

# Sanity check
images, labels = next(iter(train_loader))
print(f"Batch image shape : {images.shape}")   # [32, 3, 224, 224]
print(f"Batch label shape : {labels.shape}")   # [32]
print(f"Unique labels     : {labels.unique()}")

Batch image shape : torch.Size([32, 3, 224, 224])
Batch label shape : torch.Size([32])
Unique labels     : tensor([0, 1, 2])


In [18]:
# ──── Model & Optimizer & Scheduler & Loss ────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler()

print(f"Device: {device}")

dinov2      = AutoModel.from_pretrained("facebook/dinov2-small")           # getting the model
num_classes = len(train_dataset.classes)                                   # getting the number of actions (classes)
model       = DINOv2Classifier(dinov2, num_classes=num_classes).to(device) # setting up the model

'''
Freezing strategy:
- Epochs 1 to (UNFREEZE_LAST_N_EPOCH - 1): backbone fully frozen, only head trains
- Epochs UNFREEZE_LAST_N_EPOCH to (UNFREEZE_FULL_EPOCH - 1): unfreeze last 2 transformer blocks + layernorm
- Epochs UNFREEZE_FULL_EPOCH onwards: unfreeze entire backbone
'''
freeze_backbone(model)

optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(),   "lr": 1e-6},  # dormant until unfreezing
    {"params": model.classifier.parameters(), "lr": 1e-4},  # higher head LR at start
], weight_decay=0.05)

# linear warmup for the first 1 epoch worth of steps, then cosine decay
steps_per_epoch = len(train_loader)
warmup_steps    = steps_per_epoch  # 1 epoch warmup

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)   # linear warmup
    return 1.0                               # cosine handled by scheduler

warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=30, eta_min=1e-7
)

# weighted loss to handle class imbalance
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
scaler = torch.amp.GradScaler("cuda")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,} / {total:,}  (backbone frozen)")

# Sanity check output shape
with torch.no_grad():
    out = model(torch.randn(1, 3, 224, 224).to(device))
print(f"Output shape     : {out.shape}")  # [1, 3]

Device: cuda
Trainable params : 132,355 / 22,188,931  (backbone frozen)
Output shape     : torch.Size([1, 3])


In [19]:
def train_one_epoch(model, loader, optimizer, criterion, device, warmup_scheduler=None, step=0):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast("cuda"):
            outputs = model(images)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        if warmup_scheduler is not None:
            warmup_scheduler.step()
            step += 1

        total_loss += loss.item()
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, f1, step


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast("cuda"):
                outputs = model(images)
                loss    = criterion(outputs, labels)

            total_loss += loss.item()
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, f1

In [20]:
best_val_f1      = 0.0
epochs_no_improve = 0
warmup_step      = 0
in_warmup        = True   # warmup runs for the first epoch

history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}

epoch = 0
while epoch < MAX_EPOCHS:
    epoch += 1

    # ──── Unfreezing schedule ────
    if epoch == UNFREEZE_LAST_N_EPOCH:
        unfreeze_last_n_blocks(model, n=2)
        # Drop classifier LR now that backbone starts contributing
        for g in optimizer.param_groups:
            if g["lr"] == 1e-3:
                g["lr"] = 1e-4
        print(f"\n>>> Epoch {epoch}: Unfroze last 2 backbone blocks")

    if epoch == UNFREEZE_FULL_EPOCH:
        unfreeze_full_backbone(model)
        print(f"\n>>> Epoch {epoch}: Full backbone unfrozen")

    # Use warmup scheduler only during epoch 1
    active_warmup = warmup_scheduler if in_warmup else None

    train_loss, train_f1, warmup_step = train_one_epoch(
        model, train_loader, optimizer, criterion, device,
        warmup_scheduler=active_warmup, step=warmup_step
    )
    val_loss, val_f1 = validate(model, val_loader, criterion, device)
    torch.cuda.empty_cache()
    
    # Step cosine scheduler once per epoch (after warmup epoch)
    if not in_warmup:
        cosine_scheduler.step()
    in_warmup = False  # warmup only covers epoch 1

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)

    backbone_frozen = not any(p.requires_grad for p in model.backbone.parameters())
    status = "frozen" if backbone_frozen else "partial" if epoch < UNFREEZE_FULL_EPOCH else "full"

    print(f"Epoch {epoch}  [backbone: {status}]")
    print(f"  Train → Loss: {train_loss:.4f} | F1: {train_f1:.4f}")
    print(f"  Val   → Loss: {val_loss:.4f} | F1: {val_f1:.4f}")

    # ──── Checkpoint ────
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✓ Saved best model (Val F1: {best_val_f1:.4f})")
    else:
        epochs_no_improve += 1
        print(f"  ✗ No improvement ({epochs_no_improve}/{PATIENCE})")

    # ──── Early stopping ────
    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}.")
        break

    print()

print(f"\nTraining complete. Best Val F1: {best_val_f1:.4f}")

Epoch 1  [backbone: frozen]
  Train → Loss: 0.6114 | F1: 0.7030
  Val   → Loss: 0.5010 | F1: 0.7771
  ✓ Saved best model (Val F1: 0.7771)



Epoch 2  [backbone: frozen]
  Train → Loss: 0.2980 | F1: 0.8677
  Val   → Loss: 0.6763 | F1: 0.7177
  ✗ No improvement (1/8)



Epoch 3  [backbone: frozen]
  Train → Loss: 0.2361 | F1: 0.8985
  Val   → Loss: 0.7805 | F1: 0.7087
  ✗ No improvement (2/8)


>>> Epoch 4: Unfroze last 2 backbone blocks


Epoch 4  [backbone: partial]
  Train → Loss: 0.1785 | F1: 0.9254
  Val   → Loss: 1.0444 | F1: 0.6851
  ✗ No improvement (3/8)



Epoch 5  [backbone: partial]
  Train → Loss: 0.1365 | F1: 0.9464
  Val   → Loss: 1.3282 | F1: 0.6787
  ✗ No improvement (4/8)



Epoch 6  [backbone: partial]
  Train → Loss: 0.1115 | F1: 0.9589
  Val   → Loss: 1.6428 | F1: 0.6726
  ✗ No improvement (5/8)



Epoch 7  [backbone: partial]
  Train → Loss: 0.0969 | F1: 0.9659
  Val   → Loss: 2.6103 | F1: 0.5899
  ✗ No improvement (6/8)


>>> Epoch 8: Full backbone unfrozen


Epoch 8  [backbone: full]
  Train → Loss: 0.0839 | F1: 0.9763
  Val   → Loss: 2.7815 | F1: 0.6675
  ✗ No improvement (7/8)



Epoch 9  [backbone: full]
  Train → Loss: 0.0735 | F1: 0.9844
  Val   → Loss: 6.0554 | F1: 0.5625
  ✗ No improvement (8/8)

Early stopping triggered at epoch 9.

Training complete. Best Val F1: 0.7771


In [ ]:
# Load best checkpoint before evaluating
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        outputs = model(images.to(device))
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=train_dataset.classes,
            yticklabels=train_dataset.classes)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix — Best Checkpoint")
plt.tight_layout()
plt.show()

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="Train")
ax1.plot(history["val_loss"],   label="Val")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(history["train_f1"], label="Train")
ax2.plot(history["val_f1"],   label="Val")
ax2.set_title("Macro F1")
ax2.legend()

plt.tight_layout()
plt.show()